# Customer Support Bot – End‑to‑End Fine‑Tuning Walkthrough

This notebook demonstrates the complete pipeline:
1. Load & format the Bitext customer‑support dataset.
2. Configure QLoRA (4‑bit NF4 + LoRA) with Hugging Face 🤗 Transformers, TRL & PEFT.
3. Fine‑tune TinyLlama‑1.1B‑Chat (or any other supported model).
4. Merge the LoRA adapter and run a quick inference demo.
5. (Optional) Launch the Gradio chat UI.
6. (optional) Quantitative evaluation.

> **Tip:** Run each cell sequentially. Adjust hyper‑parameters in the "Configuration" section if you want to experiment.


# 1️⃣ Install required packages (if not already installed)

In [ ]:
%pip install -q transformers==4.40.0 trl==0.8.6 peft==0.10.0 datasets==2.18.0 bitsandbytes==0.43.0 accelerate==0.28.0 gradio==4.26.0 sentencepiece PyYAML python-multipart==0.0.9


# 2️⃣ Imports & utilities

In [ ]:
import os
import json
import torch
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from tqdm.auto import tqdm

# Helper to format examples into the chat template
def format_chat_template(example):
 return {
 "text": f"<|system|>You are a helpful customer support agent.\n\n<|user|>{example['instruction']}\n\n<|assistant|>{example['response']}"
 }


# 3️⃣ Load & prepare the dataset

In [ ]:
# Load the Bitext dataset from the Hugging Face Hub
raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(f"Number of examples: {len(raw)}")
raw[0]


In [ ]:
# Apply the chat‑template formatting
formatted = raw.map(format_chat_template, remove_columns=raw.column_names)
formatted = formatted.train_test_split(test_size=0.1, seed=42)
print(f"Train size: {len(formatted['train'])}, Validation size: {len(formatted['test'])}")
formatted['train'][0]


# 4️⃣ Model & tokenizer configuration

In [ ]:
# Choose the base model – change this to experiment with Phi‑2, Mistral, etc.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4‑bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
 load_in_4bit=True,
 bnb_4bit_quant_type="nf4",
 bnb_4bit_compute_dtype=torch.bfloat16,
 bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # ensure padding token exists

# Load model in 4‑bit
model = AutoModelForCausalLM.from_pretrained(
 model_name,
 quantization_config=bnb_config,
 device_map="auto",
 trust_remote_code=True,
)

# Prepare model for k‑bit training (adds gradient checkpointing, etc.)
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
 r=16,
 lora_alpha=32,
 target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
 lora_dropout=0.05,
 bias="none",
 task_type="CAUSAL_LM",
)

# Apply LoRA adapter
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# 5️⃣ Training arguments & SFTTrainer

In [ ]:
output_dir = "./qlora_customer_support"

training_args = TrainingArguments(
 output_dir=output_dir,
 per_device_train_batch_size=2,
 per_device_eval_batch_size=2,
 gradient_accumulation_steps=4,
 learning_rate=2e-4,
 num_train_epochs=3,
 fp16=True,
 logging_steps=10,
 evaluation_strategy="steps",
 eval_steps=200,
 save_steps=500,
 warmup_steps=100,
 lr_scheduler_type="cosine",
 optim="paged_adamw_32bit",
 push_to_hub=False,
 report_to="none",
)

trainer = SFTTrainer(
 model=model,
 train_dataset=formatted["train"],
 eval_dataset=formatted["test"],
 tokenizer=tokenizer,
 args=training_args,
 max_seq_length=512,
 packing=False, # set True if you want to pack multiple examples
)


# 6️⃣ Start fine‑tuning

In [ ]:
# Uncomment the line below to begin training (it will take a while).
# trainer.train()

# For a quick test you can run a few steps:
trainer.train()


# 7️⃣ Merge LoRA adapter into base model & save

In [ ]:
# After training, merge and save the model for easy inference.
merged_model_dir = "./merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

# Merge LoRA weights into the base model
model = model.merge_and_unload()

# Save merged model & tokenizer
model.save_pretrained(merged_model_dir)
tokenizer.save_pretrained(merged_model_dir)
print(f"Merged model saved to {merged_model_dir}")


# 8️⃣ Quick inference demo (no UI)

In [ ]:
import torch
from transformers import pipeline

# Load the merged model for generation
generator = pipeline(
 "text-generation",
 model=merged_model_dir,
 tokenizer=tokenizer,
 torch_dtype=torch.bfloat16,
 device_map="auto",
)

def generate_response(prompt, max_new_tokens=128, temperature=0.7, top_p=0.9):
 formatted_prompt = f"<|system|>You are a helpful customer support agent.\n\n<|user|>{prompt}\n\n<|assistant|>"
 outputs = generator(
 formatted_prompt,
 max_new_tokens=max_new_tokens,
 temperature=temperature,
 top_p=top_p,
 do_sample=True,
 pad_token_id=tokenizer.eos_token_id,
 )
 # Extract only the assistant's reply
 generated_text = outputs[0]["generated_text"]
 answer = generated_text.split("<|assistant|>")[-1].strip()
 return answer

# Test with a sample query
test_inst = "I cannot log into my account, what should I do?"
print("User:", test_inst)
print("Bot:", generate_response(test_inst))


# 9️⃣ (Optional) Launch Gradio chat UI

Run the following cell to start a local Gradio interface.
Make sure you have executed the training and merging steps above.


In [ ]:
import gradio as gr

# Reload tokenizer & model (merged) for the UI
ui_tokenizer = AutoTokenizer.from_pretrained(merged_model_dir)
ui_model = AutoModelForCausalLM.from_pretrained(
 merged_model_dir,
 torch_dtype=torch.bfloat16,
 device_map="auto",
)

def chat_fn(message, history, temperature=0.7, top_p=0.9, max_new_tokens=128):
 # Build prompt from history
 prompt = "<|system|>You are a helpful customer support agent.\n\n"
 for user_msg, bot_msg in history:
 prompt += f"<|user|>{user_msg}\n\n<|assistant|>{bot_msg}\n\n"
 prompt += f"<|user|>{message}\n\n<|assistant|>"

 inputs = ui_tokenizer(prompt, return_tensors="pt").to(ui_model.device)
 generated_ids = ui_model.generate(
 **inputs,
 max_new_tokens=max_new_tokens,
 temperature=temperature,
 top_p=top_p,
 do_sample=True,
 pad_token_id=ui_tokenizer.eos_token_id,
 )
 # Decode only the newly generated tokens
 generated_text = ui_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
 # Remove the prompt part
 answer = generated_text[len(prompt):].strip()
 return answer

demo = gr.ChatInterface(
 fn=chat_fn,
 additional_inputs=[
 gr.Slider(0.1, 1.0, value=0.7, label="Temperature"),
 gr.Slider(0.1, 1.0, value=0.9, label="Top-p"),
 gr.Slider(64, 512, value=128, step=64, label="Max new tokens"),
 ],
 title="Customer Support Bot (QLoRA Fine‑Tuned)",
 description="Ask any customer‑support question. The model is a QLoRA‑fine‑tuned TinyLlama‑1.1B‑Chat.",
 theme="soft",
)

demo.launch()


# 10️⃣ Quantitative evaluation (optional)

In [ ]:
from datasets import load_metric
import numpy as np

# Load a metric (e.g., ROUGE)
rouge = load_metric("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # Replace -100 (used for ignored indices) with tokenizer.pad_token_id
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE expects each prediction and reference to be a list
    result = rouge.compute(predictions=decoded_preds,
                           references=[[l] for l in decoded_labels],
                           rouge_types=["rougeL"],
                           use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# Run evaluation
eval_results = trainer.evaluate()
print("Eval loss:", eval_results["eval_loss"])
# If you added compute_metrics to the SFTTrainer, you’d see ROUGE here too
